# schemagate + LangChain: give the SQL agent only the tables this caller may read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishsinha1602/schemagate/blob/main/examples/langchain_schemagate.ipynb)

Table selection in a text-to-SQL agent runs **before** the query executes, which means it runs
before row-level security can act. An identity-blind selector hands the model a table the caller
cannot read: the SQL is correct, RLS filters every row, and the user is told *no records found* —
a wrong answer in a confident tone, not an access-denied message.

This notebook fixes the ordering. Cells 1-4 need **no API key and no database**. The LangChain
wiring at the end needs your own key.

Apache-2.0 · https://github.com/ashishsinha1602/schemagate

In [ ]:
!pip install -q schemagate langchain-community sqlalchemy

## 1. A demo catalog, with one restricted table

`hr_compensation` is readable only by callers holding the `payroll` role.

In [ ]:
from schemagate import Catalog, Principal
from schemagate.demo_schema import HINTS, create_demo_db

engine = create_demo_db()                 # in-memory SQLite, 42 objects
cat = Catalog().bootstrap(engine)
for table, hint in HINTS.items():
    cat.hint(table, hint)

cat.restrict("hr_compensation", ["payroll"])
print(f"{len(cat)} objects in the catalog")

## 2. The same question, two callers

Note what happens to `hr_compensation` — not ranked last. Absent.

In [ ]:
QUESTION = "salary and pay grade by employee"

def show(label, roles):
    p = Principal("okta:jdoe", roles=set(roles))
    sel = cat.select(QUESTION, top_k=6, principal=p)
    frag = sel.prompt_fragment()
    tables = [t.split(".")[-1] for t in sel.table_names]
    print(f"--- roles={sorted(roles) or 'none'}")
    print("  selected:", ", ".join(tables))
    print("  hr_compensation anywhere in the prompt text:", "hr_compensation" in frag)
    print(f"  prompt tokens ~{len(frag)//4}")
    return sel

no_role   = show("analyst", [])
with_role = show("payroll", ["payroll"])

## 3. What the model would actually receive

This is the entire schema text handed to the LLM for the unprivileged caller. Search it for
`hr_compensation` — the name does not appear, so the model cannot propose it, and there is no
empty result set to misread.

In [ ]:
print(no_role.prompt_fragment())

## 4. Why these tables

BM25 + a hashed embedder, fused by reciprocal rank. Offline — no model call, no API key.

In [ ]:
print(no_role.explain())

## 5. Wiring it into a LangChain SQL agent

The key line is `include_tables`: build the `SQLDatabase` **per request** from the tables
schemagate selected for *this* caller, rather than once at import from a service account's
view of the world.

Needs your own API key — the cells above do not.

In [ ]:
import os
# os.environ["OPENAI_API_KEY"] = "sk-..."

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_openai import ChatOpenAI


def agent_for(principal, question, top_k=6):
    """Build a SQL agent scoped to one caller's readable tables."""
    sel = cat.select(question, top_k=top_k, principal=principal)
    allowed = [t.split(".")[-1] for t in sel.table_names]

    db = SQLDatabase(engine, include_tables=allowed)     # <- the whole point
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    return create_sql_agent(llm, db=db, agent_type="openai-tools", verbose=True), allowed


analyst = Principal("okta:jdoe", roles=set())
payroll = Principal("okta:hr",   roles={"payroll"})

for who, p in [("analyst", analyst), ("payroll", payroll)]:
    _, allowed = agent_for(p, QUESTION)
    print(f"{who:>8}: {allowed}")

# agent, _ = agent_for(analyst, QUESTION)
# agent.invoke({"input": QUESTION})

## Notes

- Certified against a live Oracle Autonomous Database 26ai and PostgreSQL 16, not just SQLite.
- On a deliberately messy 127-object, three-domain schema, recall@6 was 47% with bare
  identifiers, 60% with Gemini 2.5 Pro writing the table descriptions, 80% with Sonnet — the
  model that *describes* your schema matters about as much as the one that writes the SQL.
- Column-level restriction: `cat.restrict_column(...)`. Audit record: `Selection.to_dict()`.
- Browser demo, no install: https://ashishsinha1602.github.io/schemagate/

`pip install schemagate` · Apache-2.0